# VAARTA - Real-Time Multilingual Voice Translation System

### By Neer Dwivedi, Shubhashish Garimella & Avichal Trivedi
**CIIC, Christ University Delhi NCR**

---

## What is Vaarta?

**Vaarta** is a real-time multilingual voice translation system designed for India's public infrastructure — hospitals, police stations, public transport, and government offices.

**The idea is simple:** Two people who don't share a language open one app, place a phone between them, speak naturally, and understand each other instantly. No buttons. No setup. No manual language selection.

---

## Supported Languages

| # | Language | Code |
|---|----------|------|
| 1 | Hindi | hi |
| 2 | Bengali | bn |
| 3 | Marathi | mr |
| 4 | Telugu | te |
| 5 | Tamil | ta |
| 6 | Gujarati | gu |
| 7 | Urdu | ur |
| 8 | Kannada | kn |
| 9 | Odia | or |
| 10 | Malayalam | ml |
| 11 | English | en |

**Total translation pairs: 110** (11 x 10 bidirectional)

---

## Technology Stack

| Layer | Technology |
|-------|-----------|
| **Backend Server** | FastAPI + Uvicorn (Python 3.13) |
| **Speech-to-Text** | faster-whisper (small model, CPU, int8) |
| **Translation** | Google Translate via deep-translator |
| **Text-to-Speech** | ElevenLabs (primary) + gTTS (fallback) |
| **Gender Detection** | librosa pitch analysis (165 Hz threshold) |
| **Frontend** | Flutter (Dart) - Android app |
| **Communication** | WebSocket (real-time bidirectional) |
| **State Management** | Provider (Flutter) |

## Project Architecture & Pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│                     FLUTTER FRONTEND (Android)                  │
│                                                                 │
│  ┌──────────┐  ┌──────────────┐  ┌───────────┐  ┌───────────┐  │
│  │  Audio   │  │  WebSocket   │  │   Home    │  │ Settings  │  │
│  │ Service  │  │   Service    │  │  Screen   │  │  Screen   │  │
│  └────┬─────┘  └──────┬───────┘  └─────┬─────┘  └─────┬─────┘  │
│       │               │               │               │         │
│       └───────────┬───┘               │               │         │
│                   │            ┌──────┴──────┐        │         │
│                   │            │  Widgets:   │        │         │
│                   │            │  Waveform   │        │         │
│                   │            │  Transcript │        │         │
│                   │            │  Clarify    │        │         │
│                   │            │  Replay     │        │         │
│                   │            └─────────────┘        │         │
└───────────────────┼───────────────────────────────────┘         │
                    │ WebSocket (ws://)                            │
                    ▼                                              │
┌─────────────────────────────────────────────────────────────────┐
│                     PYTHON BACKEND (FastAPI)                    │
│                                                                 │
│  main.py ─── Orchestrates the full pipeline:                    │
│                                                                 │
│  ┌─────────┐    ┌──────────┐    ┌───────────┐    ┌──────────┐  │
│  │  STT    │───▶│ Clarify  │───▶│ Translate │───▶│   TTS    │  │
│  │ stt.py  │    │clarify.py│    │Translate.py│   │  tts.py  │  │
│  └─────────┘    └──────────┘    └───────────┘    └──────────┘  │
│                                                                 │
│  ┌─────────────────────────────────────────────────────────┐   │
│  │  vocabulary/  →  medical.json | transport.json | personal│   │
│  └─────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────┘
```

### Pipeline Flow (per audio chunk):
1. **Microphone** captures audio on the Flutter app
2. **Audio bytes** are streamed via WebSocket to the backend
3. **STT (stt.py)** transcribes audio, detects language and speaker gender
4. **Clarify (clarify.py)** checks confidence, domain keywords, known errors
5. **Translate (Translate.py)** applies personal vocab, preserves English words, translates
6. **TTS (tts.py)** generates gender-matched speech output
7. **Result** sent back to Flutter for display and audio playback

## File Structure

```
vaarta/
├── requirements.txt                 # Python backend dependencies
├── backend/
│   ├── .env                         # API keys (ELEVENLABS_API_KEY)
│   ├── main.py                      # FastAPI server — orchestrates pipeline
│   ├── stt.py                       # Speech-to-Text engine (faster-whisper)
│   ├── Translate.py                 # Translation engine with code-switching
│   ├── tts.py                       # Text-to-Speech engine (ElevenLabs + gTTS)
│   ├── clarify.py                   # Proactive clarification engine
│   └── vocabulary/
│       ├── personal.json            # User-learned corrections (starts empty)
│       ├── medical.json             # 40+ medical domain terms
│       └── transport.json           # 30+ transport domain terms
└── frontend/
    ├── pubspec.yaml                 # Flutter dependencies
    └── lib/
        ├── main.dart                # App entry point & VaartaState
        ├── screens/
        │   ├── home_screen.dart     # Main conversation UI
        │   └── settings_screen.dart # Settings & vocabulary manager
        ├── services/
        │   ├── websocket_service.dart # Real-time WebSocket layer
        │   └── audio_service.dart     # Microphone & speaker control
        └── widgets/
            ├── waveform.dart          # Animated audio level bars
            ├── transcript_bubble.dart # Conversation message display
            ├── clarify_prompt.dart    # Clarification modal dialog
            └── replay_button.dart     # Replay last translation
```

---

# BACKEND FILES

---

## 1. `requirements.txt` — Python Dependencies

This file lists all the Python packages required by the backend server.

In [ ]:
# requirements.txt — Backend Dependencies

requirements = """
# Server
fastapi==0.111.0          # High-performance async web framework
uvicorn==0.29.0           # ASGI server to run FastAPI
python-multipart==0.0.9   # Multipart form data support
websockets==12.0          # WebSocket protocol support

# Speech to Text
faster-whisper>=1.0.1     # OpenAI Whisper optimized with CTranslate2

# Translation
deep-translator==1.11.4   # Google Translate wrapper
langdetect==1.0.9         # Language detection

# Text to Speech
elevenlabs==1.2.2         # ElevenLabs API (natural voice, gender-matched)
gTTS==2.5.1               # Google Text-to-Speech (free fallback)

# Audio Processing
pyaudio==0.2.14           # Microphone audio capture
soundfile==0.12.1         # Audio file I/O
numpy==2.4.3              # Numerical arrays for audio data
librosa==0.10.2           # Audio analysis (pitch/gender detection)

# Utilities
pydantic>=2.0.0           # Data validation (FastAPI models)
python-dotenv==1.0.1      # Load .env files
aiofiles==23.2.1          # Async file operations
httpx==0.27.0             # Async HTTP client
"""
print(requirements)

---

## 2. `backend/stt.py` — Speech-to-Text Engine

This is the **ears** of Vaarta. It takes raw audio and produces text with:
- **Language detection** — automatically identifies which of the 11 supported languages is being spoken
- **Gender detection** — uses pitch analysis (librosa) to determine male/female speaker (threshold: 165 Hz)
- **Confidence scoring** — assigns a probability score; below 75% triggers clarification
- **Silence detection** — skips silent audio chunks (amplitude threshold: 0.005)

### Key Configuration:
| Parameter | Value | Purpose |
|-----------|-------|---------|
| Model | `small` | Balance of speed and accuracy |
| Device | `cpu` | No GPU required |
| Compute Type | `int8` | Quantized for faster inference |
| Sample Rate | 16,000 Hz | Whisper's expected input format |
| Chunk Duration | 3 seconds | Amount of audio processed at a time |
| Min Confidence | 0.75 | Below this triggers clarification |

In [ ]:
# ============================================================
# stt.py — Speech to Text Engine (Complete Source)
# ============================================================
# Uses faster-whisper for real-time transcription with
# language detection and gender detection

"""
import numpy as np
import queue
import threading
import pyaudio
from faster_whisper import WhisperModel
from dataclasses import dataclass, field
from typing import Optional, Callable

# --- Configuration ---
WHISPER_MODEL_SIZE = "small"      # Options: tiny, base, small, medium, large
SAMPLE_RATE = 16000               # Whisper expects 16kHz audio
CHUNK_DURATION = 3                # Seconds of audio per transcription chunk
CHUNK_SIZE = SAMPLE_RATE * CHUNK_DURATION
SILENCE_THRESHOLD = 0.005         # Amplitude below this is considered silence
MIN_CONFIDENCE = 0.75             # Minimum confidence to accept transcription

# Supported Vaarta languages (ISO 639-1 codes)
SUPPORTED_LANGUAGES = {
    "hi": "Hindi",    "bn": "Bengali",   "mr": "Marathi",
    "te": "Telugu",   "ta": "Tamil",     "gu": "Gujarati",
    "ur": "Urdu",     "kn": "Kannada",   "or": "Odia",
    "ml": "Malayalam", "en": "English"
}


@dataclass
class TranscriptionResult:
    '''Holds the result of a single transcription chunk.'''
    text: str                        # Transcribed text
    language: str                    # Detected language code (e.g. "hi", "ta")
    language_name: str               # Human-readable language name
    confidence: float                # Confidence score (0.0 to 1.0)
    is_confident: bool               # True if confidence >= MIN_CONFIDENCE
    is_supported_language: bool      # True if language is in SUPPORTED_LANGUAGES
    gender: str = "male"             # Detected speaker gender


class SpeechToText:
    '''Real-time speech-to-text engine using faster-whisper.'''

    def __init__(self, model_size: str = WHISPER_MODEL_SIZE):
        # Load Whisper model with CPU + int8 quantization for speed
        self.model = WhisperModel(model_size, device="cpu", compute_type="int8")
        self.audio_queue = queue.Queue()
        self.is_running = False

    def _detect_gender(self, audio: np.ndarray) -> str:
        '''Detect speaker gender from audio pitch using librosa.'''
        import librosa
        pitches, magnitudes = librosa.piptrack(y=audio, sr=SAMPLE_RATE, fmin=50, fmax=400)
        pitch_values = []
        for t in range(pitches.shape[1]):
            index = magnitudes[:, t].argmax()
            pitch = pitches[index, t]
            if pitch > 0:
                pitch_values.append(pitch)
        if not pitch_values:
            return "male"
        avg_pitch = float(np.mean(pitch_values))
        # Female voices typically have higher pitch than male
        return "female" if avg_pitch > 165 else "male"

    def _is_silence(self, audio: np.ndarray) -> bool:
        '''Returns True if the audio chunk is mostly silence.'''
        return float(np.abs(audio).mean()) < SILENCE_THRESHOLD

    def _transcribe_chunk(self, audio: np.ndarray) -> Optional[TranscriptionResult]:
        '''Transcribes a single audio chunk and returns a TranscriptionResult.'''
        if self._is_silence(audio):
            return None  # Skip silent chunks

        # Transcribe with Whisper — language=None for auto-detection
        segments, info = self.model.transcribe(audio, beam_size=5, language=None, vad_filter=False)
        segments_list = list(segments)
        full_text = " ".join([s.text.strip() for s in segments_list]).strip()

        if not full_text:
            return None

        detected_lang = info.language
        lang_prob = float(info.language_probability)
        gender = self._detect_gender(audio)

        return TranscriptionResult(
            text=full_text,
            language=detected_lang,
            language_name=SUPPORTED_LANGUAGES.get(detected_lang, detected_lang.upper()),
            confidence=lang_prob,
            is_confident=lang_prob >= MIN_CONFIDENCE,
            is_supported_language=detected_lang in SUPPORTED_LANGUAGES,
            gender=gender
        )

    def transcribe_audio_data(self, audio: np.ndarray) -> Optional[TranscriptionResult]:
        '''Transcribe a single numpy audio array directly (used by main.py via WebSocket).'''
        return self._transcribe_chunk(audio)
"""
print("stt.py loaded — SpeechToText engine with Whisper + gender detection")

### How STT Works — Step by Step:

```
Raw Audio (float32, 16kHz, mono)
        │
        ▼
┌─────────────────┐
│ Silence Check   │──── amplitude < 0.005? ──── Skip chunk
│                 │
└────────┬────────┘
         ▼
┌─────────────────┐
│ Whisper Model   │  beam_size=5, auto language detection
│ (small, int8)   │  Returns: text, language, confidence
└────────┬────────┘
         ▼
┌─────────────────┐
│ Gender Detection│  librosa.piptrack → avg pitch
│ (Pitch Analysis)│  > 165 Hz = female, ≤ 165 Hz = male
└────────┬────────┘
         ▼
   TranscriptionResult
   {text, language, confidence, gender}
```

---

## 3. `backend/Translate.py` — Translation Engine with Code-Switching

This is the **brain** of Vaarta. It handles:
- **Personal vocabulary corrections** — applies user-saved corrections before translating
- **Code-switching detection** — identifies English words embedded in Indian language speech
- **English word preservation** — protects ~80 common English words from being translated
- **Pivot translation** — uses Hindi as an intermediate language when direct translation fails

### The Code-Switching Problem:
In India, people commonly mix English words into their native language speech. For example:
- *"Mujhe **hospital** jaana hai"* (Hindi + English)
- *"Naa **doctor** daggara vellali"* (Telugu + English)

If you translate the full sentence, "hospital" and "doctor" might get incorrectly translated into the target language's equivalent. Vaarta **preserves** these English words using placeholder substitution.

In [ ]:
# ============================================================
# Translate.py — Translation Engine with Code-Switching Handler
# ============================================================
# Handles translation across 10 Indian languages + English
# Intelligently preserves English words embedded in Indian language speech

"""
import re
import json
import os
from deep_translator import GoogleTranslator
from dataclasses import dataclass
from typing import Optional

SUPPORTED_LANGUAGES = {
    "hi": "Hindi",    "bn": "Bengali",   "mr": "Marathi",
    "te": "Telugu",   "ta": "Tamil",     "gu": "Gujarati",
    "ur": "Urdu",     "kn": "Kannada",   "or": "Odia",
    "ml": "Malayalam", "en": "English"
}

PIVOT_LANGUAGE = "hi"  # Hindi as fallback for rare translation pairs

# ~80 English words commonly used in Indian language speech
# These are PRESERVED as-is during translation (never translated)
UNIVERSAL_ENGLISH_WORDS = {
    # Transport
    "auto", "cab", "bike", "bus", "metro", "train", "taxi",
    "uber", "ola", "rapido", "signal", "flyover", "highway",
    "toll", "parking", "petrol", "diesel", "pump",

    # Medical
    "doctor", "hospital", "clinic", "medicine", "tablet",
    "injection", "operation", "surgery", "icu", "ot", "opd",
    "bp", "sugar", "fever", "report", "scan", "x-ray", "mri",
    "ambulance", "emergency", "ward", "bed",

    # Technology
    "phone", "mobile", "laptop", "computer", "internet",
    "wifi", "charger", "battery", "app", "online", "offline",
    "password", "otp", "upi", "gpay", "paytm",

    # Daily life, brands, cities, etc.
    "mall", "shop", "market", "hotel", "restaurant", "cafe",
    "okay", "good", "bad", "best", "cool", "awesome",
    "india", "delhi", "mumbai", "bangalore", "chennai",
    "amazon", "flipkart", "zomato", "swiggy", "youtube",
    "whatsapp", "google", "facebook", ...
}


@dataclass
class TranslationResult:
    original_text: str           # Input text
    translated_text: str         # Final translated output
    source_language: str         # Source language code
    target_language: str         # Target language code
    source_language_name: str    # e.g. "Hindi"
    target_language_name: str    # e.g. "Tamil"
    used_pivot: bool             # True if Hindi was used as intermediate
    preserved_words: list        # English words kept as-is
    applied_corrections: list    # Personal vocab corrections applied


class Translator:
    def __init__(self):
        self.personal_vocab = self._load_personal_vocab()

    def translate(self, text, source_language, target_language) -> TranslationResult:
        '''
        Main translation function — 5 step process:
        1. Apply personal vocabulary corrections
        2. Identify English words to preserve (code-switching)
        3. Replace English words with placeholders (__VAARTA_0__, etc.)
        4. Translate the protected text via Google Translate
        5. Restore English words from placeholders
        '''
        # Step 1: Apply personal vocab corrections
        corrected_text, corrections = self._apply_personal_vocab(text)

        # Step 2: Find English words to preserve
        preserved_words = self._extract_english_words(corrected_text)

        # Step 3: Protect with placeholders
        protected_text, placeholder_map = self._protect_english_words(corrected_text, preserved_words)

        # Step 4: Translate (with Hindi pivot fallback)
        translated = self._translate_text(protected_text, source_language, target_language)

        # Step 5: Restore English words
        final_text = self._restore_english_words(translated, placeholder_map)

        return TranslationResult(...)
"""

# Demonstrate the code-switching concept:
print("=== Code-Switching Example ===")
print()
print("Input:  'Mujhe hospital jaana hai, doctor se milna hai'")
print("Step 1: Personal vocab corrections → (none)")
print("Step 2: English words found → {'hospital', 'doctor'}")
print("Step 3: Protected → 'Mujhe __VAARTA_0__ jaana hai, __VAARTA_1__ se milna hai'")
print("Step 4: Translated (hi→ta) → 'எனக்கு __VAARTA_0__ போக வேண்டும், __VAARTA_1__ சந்திக்க வேண்டும்'")
print("Step 5: Restored → 'எனக்கு hospital போக வேண்டும், doctor சந்திக்க வேண்டும்'")
print()
print("✓ 'hospital' and 'doctor' preserved — not mistranslated!")

### Translation Flow:

```
Input Text: "Mujhe hospital jaana hai"
      │
      ▼
┌──────────────────────┐
│ Personal Vocab Fix   │  e.g. "doktar" → "doctor"
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ Extract English Words│  Found: {"hospital"}
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ Protect with         │  "Mujhe __VAARTA_0__ jaana hai"
│ Placeholders         │
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ Google Translate     │  hi → ta via deep-translator
│ (+ Hindi pivot       │  Falls back to Hindi as bridge
│   if direct fails)   │  for rare language pairs
└──────────┬───────────┘
           ▼
┌──────────────────────┐
│ Restore English      │  Put "hospital" back
│ Words                │
└──────────┬───────────┘
           ▼
Output: "எனக்கு hospital போக வேண்டும்"
```

---

## 4. `backend/tts.py` — Text-to-Speech Engine

This is the **voice** of Vaarta. It converts translated text back into spoken audio with:
- **Dual engine approach**: ElevenLabs (primary, high-quality) + gTTS (free fallback)
- **Gender-matched voices**: Adam (male) and Rachel (female) via ElevenLabs
- **Thread-safe playback**: Only one voice plays at a time
- **Replay capability**: Stores last spoken text for replay
- **Async speaking**: Doesn't block the pipeline during audio playback

In [ ]:
# ============================================================
# tts.py — Text to Speech Engine (Complete Source)
# ============================================================
# Primary: ElevenLabs (natural voice with gender matching)
# Fallback: gTTS (free, no API key needed)

"""
import os, io, tempfile, threading
from dotenv import load_dotenv
from gtts import gTTS
from elevenlabs import VoiceSettings
from elevenlabs.client import ElevenLabs

ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")

# Gender-matched voice IDs (ElevenLabs eleven_multilingual_v2 model)
ELEVENLABS_VOICES = {
    "male":   "pNInz6obpgDQGcFmaJgB",   # Adam
    "female": "21m00Tcm4TlvDq8ikWAM",   # Rachel
}

class TextToSpeech:
    def __init__(self):
        self.use_elevenlabs = bool(ELEVENLABS_API_KEY)
        self._playback_lock = threading.Lock()       # Thread-safe: one voice at a time
        self._last_spoken = ""                        # For replay
        self._last_language = "en"
        self._last_gender = "male"

        if self.use_elevenlabs:
            self.client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

    def _speak_elevenlabs(self, text, language, gender="male") -> bool:
        '''Generate speech using ElevenLabs with gender-matched voice.'''
        voice_id = ELEVENLABS_VOICES.get(gender, ELEVENLABS_VOICES["male"])
        audio_generator = self.client.text_to_speech.convert(
            voice_id=voice_id,
            text=text,
            model_id="eleven_multilingual_v2",
            voice_settings=VoiceSettings(
                stability=0.3,
                similarity_boost=0.8,
                style=0.5,
                use_speaker_boost=True
            )
        )
        audio_bytes = b"".join(audio_generator)
        self._play_mp3_bytes(audio_bytes)
        return True

    def _speak_gtts(self, text, language):
        '''Free fallback using Google Text-to-Speech.'''
        tts = gTTS(text=text, lang=language, slow=False)
        # Save to temp file, play, then delete
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            tts.save(f.name)
            self._play_mp3_file(f.name)
            os.unlink(f.name)

    def speak(self, text, language, gender="male"):
        '''Main speak function — tries ElevenLabs, falls back to gTTS.'''
        self._last_spoken = text    # Save for replay
        self._last_language = language
        self._last_gender = gender
        with self._playback_lock:   # Thread-safe
            if self.use_elevenlabs:
                if not self._speak_elevenlabs(text, language, gender):
                    self._speak_gtts(text, language)  # Fallback
            else:
                self._speak_gtts(text, language)

    def speak_async(self, text, language, gender="male"):
        '''Non-blocking speak — runs in a daemon thread.'''
        thread = threading.Thread(target=self.speak, args=(text, language, gender), daemon=True)
        thread.start()

    def replay(self):
        '''Replay the last spoken translation.'''
        if self._last_spoken:
            self.speak(self._last_spoken, self._last_language, self._last_gender)
"""

print("tts.py loaded — Dual TTS engine (ElevenLabs + gTTS fallback)")
print()
print("Voice Configuration:")
print("  Male voice:   Adam (ElevenLabs)")
print("  Female voice: Rachel (ElevenLabs)")
print("  Model:        eleven_multilingual_v2 (supports all 11 languages)")
print("  Stability:    0.3 | Similarity: 0.8 | Style: 0.5")

---

## 5. `backend/clarify.py` — Proactive Clarification Engine

This sits **between STT and Translation** as a safety net. In critical settings like hospitals, a mistranslation could be dangerous. The clarification engine flags potentially problematic transcriptions **before** they are translated.

### Three Trigger Conditions:

| # | Trigger | When it fires | Example |
|---|---------|--------------|---------|
| 1 | **Low Confidence** | STT confidence < 75% | Whisper isn't sure what was said |
| 2 | **Domain Keywords** | Text contains critical medical/transport terms | "injection", "operation", "surgery" |
| 3 | **Known Mangled Pattern** | Text matches a previously-corrected mistake | "doktar" (was corrected to "doctor" before) |

### Performance Requirement:
All three checks must complete in **under 500ms** to avoid noticeable delay in the pipeline.

In [ ]:
# ============================================================
# clarify.py — Proactive Clarification Engine (Complete Source)
# ============================================================
# Flags low-confidence transcriptions and domain-critical
# keywords for user confirmation before translation

"""
import json, os, time
from dataclasses import dataclass
from stt import TranscriptionResult

MIN_CONFIDENCE = 0.75

@dataclass
class ClarificationResult:
    original_text: str          # Text as received from STT
    final_text: str             # Text after clarification (original or corrected)
    was_clarified: bool         # True if clarification was triggered
    correction_saved: bool      # True if a new correction was saved
    trigger_reason: str = ""    # Why clarification was triggered


class ClarificationEngine:
    def __init__(self):
        self.medical_keywords = set()    # 40+ medical terms in 11 languages
        self.transport_keywords = set()  # 30+ transport terms in 11 languages
        self.personal_vocab = {}         # User corrections (e.g. "doktar" → "doctor")
        self._load_vocabularies()

    def check(self, result: TranscriptionResult, active_domain="general") -> ClarificationResult:
        '''
        Check transcription against all triggers. Must complete in <500ms.
        Returns ClarificationResult with was_clarified=True if any trigger fires.
        '''
        start_time = time.time()
        trigger_reason = ""

        # Trigger 1: Low confidence
        if result.confidence < MIN_CONFIDENCE:
            trigger_reason = f"low_confidence ({result.confidence:.0%})"

        # Trigger 2: Domain keywords (only in medical/transport mode)
        elif active_domain != "general":
            keywords = self.medical_keywords if active_domain == "medical" else self.transport_keywords
            for word in result.text.lower().split():
                if word in keywords:
                    trigger_reason = f"domain_keyword ({active_domain})"
                    break

        # Trigger 3: Known mangled pattern
        elif self.personal_vocab:
            for wrong in self.personal_vocab:
                if wrong.lower() in result.text.lower():
                    trigger_reason = "known_mangled_pattern"
                    break

        elapsed = time.time() - start_time
        if elapsed > 0.5:
            print(f"WARNING: Check took {elapsed:.3f}s (target <0.5s)")

        return ClarificationResult(
            original_text=result.text,
            final_text=result.text,
            was_clarified=bool(trigger_reason),
            correction_saved=False,
            trigger_reason=trigger_reason
        )

    def accept_correction(self, original_text, corrected_text, translator=None):
        '''Save user correction to personal.json and reload translator.'''
        if corrected_text.strip() != original_text.strip():
            self._save_correction(original_text.lower(), corrected_text)
            if translator:
                translator.reload_personal_vocab()
        return ClarificationResult(
            original_text=original_text,
            final_text=corrected_text or original_text,
            was_clarified=True,
            correction_saved=True
        )
"""

# Demonstrate clarification flow:
print("=== Clarification Flow Examples ===")
print()
print("Example 1: Low confidence")
print("  STT Output: 'mujhe doktar ke paas jaana hai' (60% confidence)")
print("  → TRIGGERED: low_confidence (60%)")
print("  → Frontend shows: 'Did you say: mujhe doktar ke paas jaana hai?'")
print("  → User corrects: 'mujhe doctor ke paas jaana hai'")
print("  → Saved to personal.json: {'doktar': 'doctor'}")
print()
print("Example 2: Domain keyword (medical mode)")
print("  STT Output: 'I need an injection for the fever' (92% confidence)")
print("  → TRIGGERED: domain_keyword (medical)")
print("  → 'injection' and 'fever' are critical medical terms")
print("  → Frontend asks user to confirm before translating")
print()
print("Example 3: No trigger")
print("  STT Output: 'hello how are you' (95% confidence, general mode)")
print("  → NOT triggered — proceeds directly to translation")

---

## 6. `backend/main.py` — FastAPI Server Orchestrator

This is the **central hub** that connects everything together. It:
- Creates the FastAPI application with CORS middleware
- Initializes all four pipeline components (STT, Clarify, Translate, TTS)
- Exposes REST endpoints for health checks, languages, and vocabulary management
- Manages WebSocket sessions for real-time audio streaming
- Routes audio through the full pipeline: **STT → Clarify → Translate → TTS**
- Tracks per-session state (Speaker A/B languages, gender, active domain)

### API Endpoints:

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/health` | Health check — confirms all components are ready |
| `GET` | `/languages` | List all 11 supported languages |
| `GET` | `/vocabulary` | Get personal vocabulary corrections |
| `DELETE` | `/vocabulary/{word}` | Remove a word from personal vocabulary |
| `WS` | `/ws/session` | WebSocket for real-time audio streaming |

### WebSocket Message Types:

| Type | Direction | Description |
|------|-----------|-------------|
| `session_start` | Server → Client | Connection established |
| `transcription` | Server → Client | STT result with confidence & gender |
| `clarification_request` | Server → Client | Asks user to confirm/correct text |
| `translation` | Server → Client | Final translated text |
| `speaker_switched` | Server → Client | Active speaker changed |
| `domain_changed` | Server → Client | Domain mode changed |
| `switch_speaker` | Client → Server | Toggle active speaker |
| `set_domain` | Client → Server | Change domain (general/medical/transport) |
| `clarification_response` | Client → Server | User's correction or acceptance |
| `replay` | Client → Server | Replay last TTS output |

In [ ]:
# ============================================================
# main.py — FastAPI Server Orchestrator (Key Sections)
# ============================================================

"""
# --- App Setup ---
app = FastAPI(title="Vaarta", description="Real-Time Multilingual Voice Translation System", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# --- Initialize Pipeline ---
stt_engine    = SpeechToText()          # Whisper model loaded
translator    = Translator()            # Google Translate + code-switching
tts_engine    = TextToSpeech()          # ElevenLabs + gTTS
clarify_engine = ClarificationEngine()  # Vocabulary packs loaded

# --- Session State ---
class SpeakerState:
    language: Optional[str] = None      # Detected language (auto)
    language_name: Optional[str] = None
    gender: str = "male"                # Detected gender (auto)
    last_text: str = ""

class SessionState:
    speaker_a = SpeakerState()
    speaker_b = SpeakerState()
    active_speaker: str = "A"           # Who is currently talking
    active_domain: str = "general"      # general | medical | transport


# --- Core Pipeline Function ---
async def process_audio_chunk(audio_data, session, websocket):
    '''Process a single audio chunk through the full pipeline.'''

    # Step 1: Speech to Text
    result = stt_engine.transcribe_audio_data(audio_data)
    if result is None:
        return

    # Update speaker state (language, gender)
    current_speaker.language = result.language
    current_speaker.gender = result.gender

    # Send transcription to frontend
    await websocket.send_json({"type": "transcription", ...})

    # Step 2: Clarification Check
    clarification = clarify_engine.check(result, active_domain=session.active_domain)
    if clarification.was_clarified:
        await websocket.send_json({"type": "clarification_request", ...})
        return  # Wait for user response before translating

    # Step 3: Translate to other speaker's language
    target_language = other_speaker.language or "en"
    translation = translator.translate(text, source_language, target_language)
    await websocket.send_json({"type": "translation", ...})

    # Step 4: TTS — speak the translated text (non-blocking)
    tts_engine.speak_async(translation.translated_text, language=target_language, gender=result.gender)


# --- WebSocket Endpoint ---
@app.websocket("/ws/session")
async def websocket_session(websocket):
    '''Handles real-time audio streaming for a two-speaker session.'''
    await websocket.accept()
    session = SessionState()

    while session.is_active:
        message = await websocket.receive()

        if "bytes" in message:
            # Binary audio data → process through pipeline
            audio_data = np.frombuffer(message["bytes"], dtype=np.float32)
            await process_audio_chunk(audio_data, session, websocket)

        elif "text" in message:
            # JSON command from frontend
            data = json.loads(message["text"])
            if data["type"] == "switch_speaker":
                session.active_speaker = "B" if session.active_speaker == "A" else "A"
            elif data["type"] == "set_domain":
                session.active_domain = data["domain"]
            elif data["type"] == "clarification_response":
                # User accepted or corrected → save + translate
                ...
            elif data["type"] == "replay":
                tts_engine.replay()
"""

print("main.py — FastAPI server orchestrating the full pipeline")
print()
print("Startup sequence:")
print("  1. Load Whisper model (STT)")
print("  2. Initialize Google Translator")
print("  3. Connect to ElevenLabs (TTS)")
print("  4. Load vocabulary packs (Clarify)")
print("  5. Start uvicorn on 0.0.0.0:8000")
print()
print("Runtime: WebSocket receives binary audio → processes through")
print("STT → Clarify → Translate → TTS → sends results back to Flutter")

---

## 7. `backend/vocabulary/` — Domain Vocabulary Packs

Three JSON files power the clarification and translation engines:

### `medical.json` — 40+ Medical Terms
Contains critical medical terminology in English + translations for all 11 languages:
- **Symptoms**: fever, pain, headache, vomiting, dizziness, breathlessness, swelling, bleeding
- **Body parts**: chest, stomach, head, back, arm, leg, heart, kidney, liver, lungs
- **Procedures**: injection, operation, surgery, scan, MRI, X-ray, blood test, ICU, OPD
- **Facilities**: hospital, doctor, clinic, pharmacy, ward, ambulance
- **Critical phrases**: "I need help", "I am in pain", "I cannot breathe", "call a doctor"

### `transport.json` — 30+ Transport Terms
- **Vehicles**: auto, cab, taxi, bus, metro, train, bike, rickshaw
- **Directions**: left, right, straight, stop, turn, u-turn, nearby
- **Landmarks**: signal, flyover, highway, toll, bridge, station, airport
- **Payment**: meter, fare, change, UPI, cash, receipt
- **Commands**: "stop here", "go faster", "go slower", "how far", "take me to"

### `personal.json` — Self-Learning Corrections
- Starts empty `{}`
- Grows over time as users correct misheard words
- Example after use: `{"doktar": "doctor", "febber": "fever", "hosptal": "hospital"}`
- Applied automatically during translation (via `Translate.py`)

---

# FRONTEND FILES (Flutter / Dart)

---

## 8. `frontend/pubspec.yaml` — Flutter Dependencies

The Flutter app's package manifest.

In [ ]:
# pubspec.yaml — Flutter Dependencies

pubspec = """
name: vaarta
description: Real-Time Multilingual Voice Translation System
version: 1.0.0+1

environment:
  sdk: '>=3.0.0 <4.0.0'

dependencies:
  flutter: sdk                    # Flutter framework
  web_socket_channel: ^2.4.0     # WebSocket client for real-time communication
  permission_handler: ^11.0.1    # Request microphone permissions
  record: ^5.0.4                 # Audio recording (microphone capture)
  just_audio: ^0.9.36            # Audio playback (TTS output)
  provider: ^6.1.1               # State management (ChangeNotifier)
  http: ^1.1.0                   # HTTP client for REST API calls
  path_provider: ^2.1.1          # File system paths

dev_dependencies:
  flutter_test: sdk              # Testing framework
  flutter_lints: ^3.0.1          # Lint rules
"""
print(pubspec)

---

## 9. `frontend/lib/main.dart` — App Entry Point & Central State

This file serves two purposes:

### 1. App Entry Point
- Locks orientation to portrait mode
- Sets up Material Design 3 theme (cream background `#F8F7F4`, dark text `#1A1A1A`)
- Wraps the app in `MultiProvider` for state management
- Defines routes: `/` (HomeScreen) and `/settings` (SettingsScreen)

### 2. VaartaState (ChangeNotifier)
The **central state manager** for the entire app. It:
- Manages WebSocket and audio service lifecycle
- Tracks connection status, active domain, active speaker
- Stores conversation transcripts as `TranscriptEntry` objects
- Handles all incoming WebSocket messages (transcription, translation, clarification)
- Provides actions: `connect()`, `disconnect()`, `switchSpeaker()`, `setDomain()`, `replay()`
- Manages clarification flow: `acceptClarification()`, `correctClarification()`, `dismissClarification()`

In [ ]:
# ============================================================
# main.dart — App Entry Point & VaartaState (Complete Source)
# ============================================================

main_dart = """
// Entry point — locks portrait, sets up providers and theme
void main() {
  WidgetsFlutterBinding.ensureInitialized();
  SystemChrome.setPreferredOrientations([DeviceOrientation.portraitUp]);
  runApp(const VaartaApp());
}

class VaartaApp extends StatelessWidget {
  Widget build(BuildContext context) {
    return MultiProvider(
      providers: [ChangeNotifierProvider(create: (_) => VaartaState())],
      child: MaterialApp(
        title: 'Vaarta',
        theme: ThemeData(
          scaffoldBackgroundColor: Color(0xFFF8F7F4),  // Cream
          colorScheme: ColorScheme.fromSeed(seedColor: Color(0xFF1A1A1A)),
          fontFamily: 'Roboto',
          useMaterial3: true,
        ),
        home: HomeScreen(),
        routes: {'/settings': (context) => SettingsScreen()},
      ),
    );
  }
}

/// Central app state — manages WebSocket, audio, transcriptions, and domain.
class VaartaState extends ChangeNotifier {
  final WebSocketService _wsService = WebSocketService();
  final AudioService _audioService = AudioService();

  // State variables
  bool _isConnected = false;
  String _activeDomain = 'general';       // general | medical | transport
  String _activeSpeaker = 'A';            // A or B
  List<TranscriptEntry> _transcripts = [];
  bool _showClarification = false;
  String _clarificationText = '';
  double _speakerALevel = 0.0;            // Audio level for waveform
  double _speakerBLevel = 0.0;
  String _serverUrl = '10.0.2.2:8000';    // Android emulator default

  // Connect to backend and start audio streaming
  Future<void> connect({String? serverUrl}) async {
    await _wsService.connect('ws://\$_serverUrl/ws/session');
    await _audioService.initialize();
    _audioService.startRecording((audioBytes) {
      if (_isConnected) _wsService.sendAudioBytes(audioBytes);
    });
  }

  // Handle incoming WebSocket messages
  void _handleMessage(Map<String, dynamic> message) {
    switch (message['type']) {
      case 'transcription':   // Update waveform levels
      case 'translation':     // Add transcript entry
      case 'clarification_request':  // Show clarification prompt
      case 'speaker_switched':       // Update active speaker
      case 'domain_changed':         // Update domain
    }
    notifyListeners();  // Rebuild UI
  }

  // User actions
  void switchSpeaker() => _wsService.sendJson({'type': 'switch_speaker'});
  void setDomain(String d) => _wsService.sendJson({'type': 'set_domain', 'domain': d});
  void replay() => _wsService.sendJson({'type': 'replay'});
  void acceptClarification() => _wsService.sendJson({...});
  void correctClarification(String c) => _wsService.sendJson({...});
}

/// A single transcript entry in the conversation.
class TranscriptEntry {
  final String speaker;           // "A" or "B"
  final String originalText;      // What was said (original language)
  final String translatedText;    // Translated version
  final String sourceLanguage;    // e.g. "Hindi"
  final String targetLanguage;    // e.g. "Tamil"
  final DateTime timestamp;
}
"""
print("main.dart — App entry point + VaartaState (central state manager)")
print()
print("State flow:")
print("  VaartaState (ChangeNotifier)")
print("    ├── WebSocketService  → sends/receives messages")
print("    ├── AudioService      → captures mic, plays audio")
print("    ├── _transcripts[]    → conversation history")
print("    ├── _activeDomain     → general/medical/transport")
print("    ├── _activeSpeaker    → A or B")
print("    └── notifyListeners() → triggers UI rebuild")

---

## 10. `frontend/lib/services/websocket_service.dart` — WebSocket Communication

The **real-time communication layer** between the Flutter app and the Python backend.

### Key Features:
- **Auto-reconnect** with exponential backoff (max 10 attempts, 3-second delay)
- **Binary + Text support**: sends audio as raw bytes, commands as JSON
- **Callbacks**: `onMessage` for incoming data, `onConnectionChanged` for status updates
- Graceful error handling and cleanup

In [ ]:
# ============================================================
# websocket_service.dart — WebSocket Communication Layer
# ============================================================

websocket_dart = """
class WebSocketService {
  WebSocketChannel? _channel;
  bool _isConnected = false;
  int _reconnectAttempts = 0;
  static const int _maxReconnectAttempts = 10;
  static const Duration _reconnectDelay = Duration(seconds: 3);

  // Callbacks
  void Function(Map<String, dynamic>)? onMessage;
  void Function(bool)? onConnectionChanged;

  // Connect to WebSocket server
  Future<void> connect(String url) async {
    _channel = WebSocketChannel.connect(Uri.parse(url));
    await _channel!.ready;
    _isConnected = true;
    onConnectionChanged?.call(true);

    // Listen for incoming data
    _channel!.stream.listen(
      (data) {
        if (data is String) {
          // JSON text message → parse and forward
          onMessage?.call(jsonDecode(data));
        } else if (data is Uint8List) {
          // Binary audio data from server
          onMessage?.call({'type': 'audio_data', 'bytes': data});
        }
      },
      onError: (_) => _scheduleReconnect(),
      onDone: () => _scheduleReconnect(),
    );
  }

  // Send JSON command to server
  void sendJson(Map<String, dynamic> data) {
    _channel?.sink.add(jsonEncode(data));
  }

  // Send raw audio bytes to server
  void sendAudioBytes(Uint8List bytes) {
    _channel?.sink.add(bytes);
  }

  // Auto-reconnect with backoff
  void _scheduleReconnect() {
    if (_reconnectAttempts >= _maxReconnectAttempts) return;
    Timer(_reconnectDelay, () {
      _reconnectAttempts++;
      _connect();  // Retry connection
    });
  }
}
"""
print("websocket_service.dart — Real-time bidirectional communication")
print()
print("Data flow:")
print("  Flutter App  ──── audio bytes ────►  Python Backend")
print("  Flutter App  ──── JSON commands ──►  Python Backend")
print("  Flutter App  ◄─── JSON results ────  Python Backend")
print("  Flutter App  ◄─── audio bytes  ────  Python Backend")
print()
print("Reconnection: up to 10 attempts with 3-second delays")

---

## 11. `frontend/lib/services/audio_service.dart` — Microphone & Speaker

Handles **audio capture** (microphone → WebSocket) and **audio playback** (server → speaker).

### Recording Configuration:
| Parameter | Value |
|-----------|-------|
| Encoder | PCM 16-bit |
| Sample Rate | 16,000 Hz |
| Channels | Mono (1) |
| Bit Rate | 256 kbps |

### Key Methods:
- `initialize()` — requests microphone permission
- `startRecording(callback)` — streams audio chunks via callback to WebSocket
- `stopRecording()` — stops capture and cleans up
- `playAudioBytes(bytes)` — plays MP3 audio from server (TTS output)

In [ ]:
# ============================================================
# audio_service.dart — Microphone Capture & Audio Playback
# ============================================================

audio_dart = """
class AudioService {
  final AudioRecorder _recorder = AudioRecorder();  // record package
  final AudioPlayer _player = AudioPlayer();        // just_audio package

  Future<void> initialize() async {
    // Request microphone permission from Android
    final status = await Permission.microphone.request();
    if (!status.isGranted) return;
  }

  Future<void> startRecording(void Function(Uint8List) onAudioData) async {
    // Start streaming audio from microphone
    final stream = await _recorder.startStream(
      RecordConfig(
        encoder: AudioEncoder.pcm16bits,   // Raw PCM format
        sampleRate: 16000,                 // Whisper's expected rate
        numChannels: 1,                    // Mono
        bitRate: 256000,                   // 256 kbps
      ),
    );

    // Each audio chunk is sent to WebSocket in real-time
    stream.listen((data) => onAudioData(data));
  }

  Future<void> playAudioBytes(Uint8List audioBytes) async {
    // Play MP3 audio received from server (TTS output)
    final dataUri = 'data:audio/mp3;base64,\${base64Encode(audioBytes)}';
    await _player.setUrl(dataUri);
    await _player.play();
  }

  void dispose() {
    _recorder.dispose();
    _player.dispose();
  }
}
"""
print("audio_service.dart — Microphone capture + audio playback")
print()
print("Audio flow:")
print("  Microphone → PCM 16-bit, 16kHz, mono → WebSocket → Backend")
print("  Backend → MP3 audio → just_audio player → Speaker")

---

## 12. `frontend/lib/screens/home_screen.dart` — Main Conversation UI

The primary user interface. It has two states:

### Connection Screen (before connecting):
- "VAARTA" title with description
- Server address input field (default: `10.0.2.2:8000`)
- "Connect" button
- Creator credits

### Main Conversation Screen (after connecting):
- **Top bar**: Domain selector chips (General / Medical / Transport) + Settings button
- **Split layout**: Screen divided into two halves for Speaker A and Speaker B
  - Each half shows: speaker label, animated waveform, scrollable transcript list
  - Tap to switch active speaker (highlighted with subtle background)
- **Overlays**: Clarification prompt (modal), Replay button (floating), Connection status indicator

In [ ]:
# ============================================================
# home_screen.dart — Main Conversation Screen (Key Structure)
# ============================================================

home_dart = """
class HomeScreen extends StatefulWidget { ... }

class _HomeScreenState extends State<HomeScreen> {
  // Before connection: show server input screen
  // After connection: show split conversation screen

  Widget build(BuildContext context) {
    return Consumer<VaartaState>(
      builder: (context, state, _) {
        if (!state.isConnected) return _buildConnectionScreen();

        return Scaffold(
          body: Stack(children: [
            Column(children: [
              // Top bar: [General] [Medical] [Transport]  ⚙️
              _buildTopBar(state),

              // Speaker A zone (top 50%) — tap to activate
              Expanded(child: GestureDetector(
                onTap: () => state.switchSpeaker(),
                child: Container(
                  color: state.activeSpeaker == 'A'
                      ? Color(0xFFF0EDE8)    // Highlighted
                      : Color(0xFFF8F7F4),   // Normal
                  child: Column(children: [
                    Text('Speaker A'),
                    WaveformWidget(level: state.speakerALevel),
                    Expanded(child: _buildTranscriptList(state, 'A')),
                  ]),
                ),
              )),

              Divider(),

              // Speaker B zone (bottom 50%) — tap to activate
              Expanded(child: GestureDetector(
                onTap: () => state.switchSpeaker(),
                child: Container(
                  // Same structure as Speaker A
                  child: Column(children: [
                    Text('Speaker B'),
                    WaveformWidget(level: state.speakerBLevel),
                    Expanded(child: _buildTranscriptList(state, 'B')),
                  ]),
                ),
              )),
            ]),

            // Overlays
            if (state.showClarification) ClarifyPrompt(...),
            Positioned(bottom: 24, right: 24, child: ReplayButton()),
            if (!state.isConnected) _connectionStatusBadge(),
          ]),
        );
      },
    );
  }
}

// Domain selector chip widget
class _DomainChip extends StatelessWidget {
  // Dark background when active, outlined when inactive
  // Text: white when active, grey when inactive
}
"""
print("home_screen.dart — Main conversation UI")
print()
print("Screen layout:")
print("  ┌─────────────────────────────────┐")
print("  │ [General] [Medical] [Transport] ⚙│  ← Domain chips + Settings")
print("  ├─────────────────────────────────┤")
print("  │         Speaker A               │")
print("  │    ▁▂▃▅▇▅▃▂▁ (waveform)        │  ← Tap to activate")
print("  │    'original text'              │")
print("  │    TRANSLATED TEXT              │")
print("  │    Hindi → Tamil                │")
print("  ├─────────────────────────────────┤")
print("  │         Speaker B               │")
print("  │    ▁▂▃▅▇▅▃▂▁ (waveform)        │  ← Tap to activate")
print("  │    'original text'              │")
print("  │    TRANSLATED TEXT              │")
print("  │    Tamil → Hindi            [⟳] │  ← Replay button")
print("  └─────────────────────────────────┘")

---

## 13. `frontend/lib/screens/settings_screen.dart` — Settings & Vocabulary Manager

A secondary screen accessible via the settings icon. Contains three sections:

1. **Active Domain Selector** — toggle between General, Medical, Transport
2. **Personal Vocabulary** — view/delete saved corrections (fetched via REST API)
3. **About** — app title, creators, stats (11 languages, 110 pairs)

In [ ]:
# ============================================================
# settings_screen.dart — Settings & Vocabulary Manager
# ============================================================

settings_dart = """
class SettingsScreen extends StatefulWidget { ... }

class _SettingsScreenState extends State<SettingsScreen> {
  Map<String, dynamic> _vocabulary = {};
  bool _isLoading = true;

  void initState() {
    _loadVocabulary();  // Fetch from GET /vocabulary
  }

  Future<void> _loadVocabulary() async {
    final response = await http.get(Uri.parse('http://\$serverUrl/vocabulary'));
    _vocabulary = jsonDecode(response.body)['vocabulary'];
  }

  Future<void> _deleteWord(String word) async {
    await http.delete(Uri.parse('http://\$serverUrl/vocabulary/\$word'));
    _vocabulary.remove(word);
  }

  Widget build(context) {
    return Scaffold(
      appBar: AppBar(title: Text('Settings')),
      body: ListView(children: [
        // Section 1: Domain selector
        _buildSectionHeader('Active Domain'),
        Row(children: [
          _buildDomainOption('general', 'General'),
          _buildDomainOption('medical', 'Medical'),
          _buildDomainOption('transport', 'Transport'),
        ]),

        // Section 2: Personal vocabulary list
        _buildSectionHeader('Personal Vocabulary'),
        // Shows: "wrong" (strikethrough, red) → "correct" (bold) [delete]
        // e.g.:  doktar → doctor  [🗑️]
        //        febber → fever   [🗑️]

        // Section 3: About
        _buildSectionHeader('About'),
        // VAARTA - Real-Time Multilingual Voice Translation
        // by Neer Dwivedi, Shubhashish Garimella & Avichal Trivedi
        // 11 languages | 110 translation pairs | Real-time
      ]),
    );
  }
}
"""
print("settings_screen.dart — Settings and vocabulary management")
print()
print("Vocabulary display format:")
print("  ┌─────────────────────────────────┐")
print("  │  doktar  →  doctor          [x] │")
print("  │  febber  →  fever           [x] │")
print("  │  hosptal →  hospital        [x] │")
print("  └─────────────────────────────────┘")
print()
print("REST API calls:")
print("  GET    /vocabulary        → load all corrections")
print("  DELETE /vocabulary/{word} → remove a correction")

---

## 14-17. `frontend/lib/widgets/` — UI Components

Four reusable widgets that make up the conversation interface:

### `waveform.dart` — Animated Audio Level Visualizer
- 32 vertical bars representing audio frequency spectrum
- Responds to confidence level (0.0 to 1.0) from STT
- Uses `CustomPainter` with sine wave pattern + random noise
- Dark bars when active (speaking), light grey when idle
- Repeating animation every 1.5 seconds

### `transcript_bubble.dart` — Conversation Message Display
- Shows original text (small, grey, 12px)
- Shows translated text (large, bold, 18px)
- Shows language pair indicator (tiny, 10px) e.g. "Hindi → Tamil"

### `clarify_prompt.dart` — Clarification Modal Dialog
- Appears as an overlay when clarification is triggered
- Shows "Did you say:" with the flagged text
- Text input field for user to type correction
- "Yes, that's correct" accept button
- Fade-in animation (300ms)

### `replay_button.dart` — Replay Last Translation
- 52x52 dark circular floating button (bottom-right)
- Replay icon (changes to volume icon while playing)
- Prevents multiple simultaneous replays
- Auto-resets after 2 seconds

In [ ]:
# ============================================================
# Widget Source Code Highlights
# ============================================================

# --- waveform.dart ---
waveform = """
class WaveformWidget extends StatefulWidget {
  final double level;  // 0.0 to 1.0

  // Uses AnimationController repeating every 1.5 seconds
  // CustomPainter draws 32 vertical bars
  // Bar height = f(sine_wave, random_noise, confidence_level)
  // Active (level > 0.05): responsive, dark bars
  // Idle (level <= 0.05): subtle ambient animation, grey bars
}
"""

# --- transcript_bubble.dart ---
bubble = """
class TranscriptBubble extends StatelessWidget {
  final TranscriptEntry entry;

  Widget build(context) {
    return Column(children: [
      Text(entry.originalText,    style: {fontSize: 12, color: grey}),
      Text(entry.translatedText,  style: {fontSize: 18, fontWeight: bold}),
      Text('\${entry.sourceLanguage} → \${entry.targetLanguage}',
           style: {fontSize: 10, color: lightGrey}),
    ]);
  }
}
"""

# --- clarify_prompt.dart ---
clarify = """
class ClarifyPrompt extends StatefulWidget {
  final String text;           // Flagged text
  final String reason;         // Why it was flagged
  final VoidCallback onAccept;
  final Function(String) onCorrect;
  final VoidCallback onDismiss;

  // 300ms fade-in animation
  // Shows: "Did you say: [flagged text]"
  // TextField for correction + send button
  // "Yes, that's correct" accept button
}
"""

# --- replay_button.dart ---
replay = """
class ReplayButton extends StatefulWidget {
  // 52x52 dark circular button
  // Tap → sends 'replay' command via WebSocket
  // Icon: replay (idle) / volume_up (playing)
  // 200ms color animation during playback
  // Auto-disables for 2 seconds after tap
}
"""

print("Widget summary:")
print("  WaveformWidget    — 32-bar animated audio visualizer")
print("  TranscriptBubble  — original + translated text display")
print("  ClarifyPrompt     — modal dialog for user correction")
print("  ReplayButton      — floating button to replay last TTS")

---

# Key Innovation Summary

## What Makes Vaarta Unique:

| Feature | How It Works | Why It Matters |
|---------|-------------|----------------|
| **Zero-Setup Language Detection** | Whisper auto-detects language from audio | Users never select a language manually |
| **Code-Switching Intelligence** | Placeholder substitution for ~80 English words | Prevents mistranslation of borrowed English words in Indian languages |
| **Proactive Clarification** | 3-trigger system (confidence, domain, history) | Prevents dangerous mistranslations in hospitals/transport |
| **Self-Learning Vocabulary** | Corrections saved to `personal.json`, auto-applied | System gets smarter with each use |
| **Gender-Matched Voice** | Pitch analysis (165 Hz threshold) → voice selection | Natural-sounding output that matches the speaker |
| **Dual TTS Fallback** | ElevenLabs primary, gTTS fallback | Works even without an API key |
| **Domain-Aware Processing** | Medical (40+ terms) and Transport (30+ terms) vocab packs | Extra safety for critical terminology |
| **Real-Time Streaming** | WebSocket + async pipeline + non-blocking TTS | Conversation flows naturally without pauses |

---

## How to Run

### Backend:
```bash
cd backend
pip install -r ../requirements.txt
# Add your ElevenLabs API key to .env (optional — gTTS works without it)
python main.py
# Server starts on http://0.0.0.0:8000
```

### Frontend:
```bash
cd frontend
flutter pub get
flutter run
# Enter server address (e.g., 192.168.x.x:8000 for physical device)
```

---

*VAARTA — Breaking language barriers in India's public infrastructure.*
*By Neer Dwivedi, Shubhashish Garimella & Avichal Trivedi | CIIC, Christ University Delhi NCR*